In [1]:
%cd /glade/derecho/scratch/lizhili/m2l8/sr_model_code/BIDiff_M2L8

/glade/derecho/scratch/lizhili/m2l8/sr_model_code/BIDiff_M2L8


/glade/derecho/scratch/lizhili/BiDiff/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import torch.nn.functional as F
import tensorflow as tf
import numpy as np
import torch.nn as nn

2026-01-25 14:41:08.133892: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"  # Use only GPU 1

In [4]:
from basicsr.utils.options import ordered_yaml
import yaml
file_load = 'options/train/train_BI_DiffSR_x4.yml'
with open(file_load) as f:
    opt = yaml.load(f, Loader=ordered_yaml()[0])

In [5]:
opt['is_train'] = True
opt['scale']=16
opt['dist']=False

In [6]:
from diffglv.models.BI_DiffSR_model import BIDiffSRModel
model = BIDiffSRModel(opt)

/glade/derecho/scratch/lizhili/BiDiff/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/glade/derecho/scratch/lizhili/BiDiff/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/glade/derecho/scratch/lizhili/BiDiff/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /glade/derecho/scratch/lizhili/BiDiff/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth


In [7]:
def input_pipeline(filename, batch_size, is_shuffle=True, is_train=True, is_repeat=True):
    feature_description = {
        'lres': tf.io.FixedLenFeature([60*60*7], dtype=tf.int64),
        'hres': tf.io.FixedLenFeature([1000*1000*7], dtype=tf.int64),
    }

    def _parse_function(example_proto):
        feature_dict = tf.io.parse_single_example(example_proto, feature_description)
        lres_img = tf.reshape(feature_dict['lres'], [7, 60, 60])
        hres_img = tf.reshape(feature_dict['hres'], [7, 1000, 1000])

        return lres_img, hres_img

    def _augment_function(lres_img, hres_img):
    # Transpose to [H, W, C]
        lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
        hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]

        # Randomly choose 0, 90, 180, or 270 degrees
        k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)

        # Apply rotation
        lres_img = tf.image.rot90(lres_img, k=k)
        hres_img = tf.image.rot90(hres_img, k=k)

        # Transpose back to [C, H, W]
        lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
        hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]

        return lres_img, hres_img

    dataset = tf.data.TFRecordDataset(filename)
    if is_repeat:
        dataset = dataset.repeat()
    dataset = dataset.map(_parse_function)
    if is_train:
        dataset = dataset.map(_augment_function)
    if is_shuffle:
        dataset = dataset.shuffle(buffer_size=100)
    batch = dataset.batch(batch_size=batch_size)
    return batch

# filenames = ['/content/drive/MyDrive/GeoSR/L8MODIS_30000_2/L8MODIS.tfrecords']
# ds = input_pipeline(filenames, batch_size=5, is_shuffle=False, is_train=True, is_repeat=True)

# for lres_batch, hres_batch in ds:
#     print(lres_batch.shape)
#     print(hres_batch.shape)

#     for i in range(lres_batch.shape[0]):
#         fig, axes = plt.subplots(1, 2, figsize=(8, 4))

#         lres_img = np.transpose(lres_batch[i].numpy(), (1, 2, 0))/3500.0
#         hres_img = np.transpose(hres_batch[i].numpy(), (1, 2, 0))/255.0

#         axes[0].imshow(lres_img[:, :, 3:0:-1])
#         axes[0].set_title('Low-Resolution')
#         axes[0].axis('off')

#         axes[1].imshow(hres_img[:, :, :3])
#         axes[1].set_title('High-Resolution')
#         axes[1].axis('off')

#         plt.show()

#     break


In [ ]:
def load_matched_weights(model, checkpoint_path):
    # Load checkpoint (state_dict)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint
    # state_dict = state_dict['params']

    # Filter only matching keys
    model_dict = model.state_dict()
    matched_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.shape == model_dict[k].shape}

    # Load the matched weights
    model_dict.update(matched_dict)
    model.load_state_dict(model_dict)

    print(f"✅ Loaded {len(matched_dict)} matching parameters out of {len(model_dict)} total.")

    return model

# model.net_g = load_matched_weights(model.net_g, '/content/drive/MyDrive/GeoSR_new/M2L8/SR_pretrained_weights/BiDiff_M2L8_x4_weights_finetune.pth')
# model.net_g = load_matched_weights(model.net_g, '/glade/derecho/scratch/lizhili/m2l8/SR_finetuned_weights/BiDiff_M2L8_x4_weights_finetune_new.pth')
model.net_g = load_matched_weights(model.net_g, '/glade/derecho/scratch/lizhili/m2l8/BiDiff_M2L8_x4_weights_finetune_new.pth')

In [9]:
def random_crop_pair(lr, hr, crop_size):
    """
    lr, hr: tensors of shape [B, C, H, W]
    crop_size: int
    """
    _, _, H, W = lr.shape
    assert H >= crop_size and W >= crop_size

    top = torch.randint(0, H - crop_size + 1, (1,)).item()
    left = torch.randint(0, W - crop_size + 1, (1,)).item()

    lr_crop = lr[:, :, top:top+crop_size, left:left+crop_size]
    hr_crop = hr[:, :, top:top+crop_size, left:left+crop_size]

    return lr_crop, hr_crop

# SR Training

In [ ]:
import torch.nn.functional as F
import tensorflow as tf
import torch

filenames = ['/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_0.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_1.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_2.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_3.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_4.tfrecords']

ds = input_pipeline(filenames, batch_size=2, is_shuffle=True, is_train=True, is_repeat=False)


    
for epoch in range(30):
    print(f'Epoch {epoch}')

    for step, (lr, hr) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
        hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')*0.0000275-0.2
        # lr = F.interpolate(lr, size=(1024, 1024), mode='nearest')
        # hr = F.interpolate(hr, size=(1024, 1024), mode='nearest')
        lr = F.interpolate(lr, size=(1024, 1024), mode='bilinear', align_corners=False)
        hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)
        lr, hr = random_crop_pair(lr, hr, crop_size=512)

        lr = (lr-0.5)/0.5
        hr = (hr-0.5)/0.5

        current_iter = step+1
        model.update_learning_rate(current_iter, warmup_iter=opt['train'].get('warmup_iter', -1))
        model.feed_data({'lq': lr, 'gt': hr})
        model.optimize_parameters(current_iter)

        if step % 500 == 0:
            print(f'Step {step}, Loss: {model.get_current_log()}')

    torch.save(model.net_g.state_dict(), '/glade/derecho/scratch/lizhili/m2l8/BiDiff_M2L8_x16_weights_finetune_new.pth')

Epoch 0


2026-01-22 21:06:43.176006: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:381] TFRecordDataset `buffer_size` is unspecified, default to 262144


Step 0, Loss: OrderedDict([('l_pix', 0.009166660718619823)])
Step 500, Loss: OrderedDict([('l_pix', 0.015475966036319733)])
Step 1000, Loss: OrderedDict([('l_pix', 0.008594944141805172)])
Step 1500, Loss: OrderedDict([('l_pix', 0.024831078946590424)])
Step 2000, Loss: OrderedDict([('l_pix', 0.0037656184285879135)])
Step 2500, Loss: OrderedDict([('l_pix', 0.011789616197347641)])
Step 3000, Loss: OrderedDict([('l_pix', 0.004416041076183319)])
Step 3500, Loss: OrderedDict([('l_pix', 0.00775873800739646)])
Step 4000, Loss: OrderedDict([('l_pix', 0.02460329420864582)])
Step 4500, Loss: OrderedDict([('l_pix', 0.058552250266075134)])
Step 5000, Loss: OrderedDict([('l_pix', 0.006005655508488417)])


2026-01-22 21:53:17.207878: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 1
Step 0, Loss: OrderedDict([('l_pix', 0.0033319813665002584)])
Step 500, Loss: OrderedDict([('l_pix', 0.004982817452400923)])
Step 1000, Loss: OrderedDict([('l_pix', 0.004773459397256374)])
Step 1500, Loss: OrderedDict([('l_pix', 0.003926876932382584)])
Step 2000, Loss: OrderedDict([('l_pix', 0.011209503747522831)])
Step 2500, Loss: OrderedDict([('l_pix', 0.007518096826970577)])
Step 3000, Loss: OrderedDict([('l_pix', 0.0207542572170496)])
Step 3500, Loss: OrderedDict([('l_pix', 0.03835000470280647)])
Step 4000, Loss: OrderedDict([('l_pix', 0.0031360473949462175)])
Step 4500, Loss: OrderedDict([('l_pix', 0.01972440630197525)])
Step 5000, Loss: OrderedDict([('l_pix', 0.03282368928194046)])


2026-01-22 22:39:51.484079: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 2
Step 0, Loss: OrderedDict([('l_pix', 0.07942947000265121)])
Step 500, Loss: OrderedDict([('l_pix', 0.13807491958141327)])
Step 1000, Loss: OrderedDict([('l_pix', 0.07415403425693512)])
Step 1500, Loss: OrderedDict([('l_pix', 0.013061200268566608)])
Step 2000, Loss: OrderedDict([('l_pix', 0.03700418397784233)])
Step 2500, Loss: OrderedDict([('l_pix', 0.019337067380547523)])
Step 3000, Loss: OrderedDict([('l_pix', 0.014604824595153332)])
Step 3500, Loss: OrderedDict([('l_pix', 0.022854706272482872)])
Step 4000, Loss: OrderedDict([('l_pix', 0.004452368710190058)])
Step 4500, Loss: OrderedDict([('l_pix', 0.0036045261658728123)])
Step 5000, Loss: OrderedDict([('l_pix', 0.0225518811494112)])
Epoch 3
Step 0, Loss: OrderedDict([('l_pix', 0.039379000663757324)])
Step 500, Loss: OrderedDict([('l_pix', 0.006231694482266903)])
Step 1000, Loss: OrderedDict([('l_pix', 0.6315324306488037)])
Step 1500, Loss: OrderedDict([('l_pix', 0.09165219217538834)])
Step 2000, Loss: OrderedDict([('l_pix', 

2026-01-23 00:12:37.730630: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


# Downstream SR Finetuning

In [ ]:
def downstream_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    finetune_tfrecords,
                    finetuned_model,
                    output_tfrecords,
                    model):
    model.net_g = load_matched_weights(model.net_g, '/glade/derecho/scratch/lizhili/m2l8/BiDiff_M2L8_x16_weights_finetune_new.pth')
    # model.net_g = load_matched_weights(model.net_g, finetuned_model)

    num_test = num_sample-num_training


    def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([7*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([7*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [7, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [7, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            return lres, hres, label

        @tf.function
        def _augment_function(lres_img, hres_img, label):
        # Transpose to [H, W, C]
            lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
            hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
    
            # Randomly choose 0, 90, 180, or 270 degrees
            k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    
            # Apply rotation
            lres_img = tf.image.rot90(lres_img, k=k)
            hres_img = tf.image.rot90(hres_img, k=k)
            label = tf.image.rot90(label, k=k)
    
            # Transpose back to [C, H, W]
            lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
            hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
    
            return lres_img, hres_img, label

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)
        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_train:
            dataset = dataset.map(_augment_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch


    # --------------------------------------------
    print('Begin Finetune')
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 2, 0, num_training, is_shuffle=True, is_train=True, is_repeat=False)

    for epoch in range(10):
        print(f'Epoch {epoch}')

        for step, (lr, hr, _) in enumerate(ds):
            lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
            hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')*0.0000275-0.2
            lr = F.interpolate(lr, size=(1024, 1024), mode='bilinear', align_corners=False)
            hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)
            lr, hr = random_crop_pair(lr, hr, crop_size=512)

            lr = (lr-0.5)/0.5
            hr = (hr-0.5)/0.5


            current_iter = step+1
            model.update_learning_rate(current_iter, warmup_iter=opt['train'].get('warmup_iter', -1))
            model.feed_data({'lq': lr, 'gt': hr})
            model.optimize_parameters(current_iter)

            if step % 500 == 0:
                print(f'Step {step}, Loss: {model.get_current_log()}')

        torch.save(model.net_g.state_dict(), finetuned_model)

    # --------------------------------------------
    print('Begin Write SR dataset')
    # TFRecord writer setup
    writer = tf.io.TFRecordWriter(output_tfrecords)

    # Serialization function
    def serialize_example(hres, label):
        feature = {
            'hres': tf.train.Feature(int64_list=tf.train.Int64List(value=hres.reshape(-1))),
            'label': tf.train.Feature(int64_list=tf.train.Int64List(value=label.reshape(-1)))
        }
        example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
        return example_proto.SerializeToString()

    # Inference loop
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 4, 0, num_sample, is_shuffle=False, is_train=False, is_repeat=False)
    for step, (lr, hr, label) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
        hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')*0.0000275-0.2

        lr = F.interpolate(lr, size=(1024, 1024), mode='bilinear', align_corners=False)
        hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)

        lr = (lr-0.5)/0.5
        hr = (hr-0.5)/0.5

        model.feed_data({'lq': lr, 'gt': hr})
        model.test()
        output = model.get_current_visuals()['result']
        output = output.detach().cpu().numpy()
        output = np.clip(output*0.5+0.5, 0, 1)
        output = ((output+0.2)/0.0000275).astype(int)
        output[output<0] = 0
        label = label.numpy()

        if step == 0:
            lr = lr.detach().cpu().numpy()

        print('batch: ', step)

        for i in range(output.shape[0]):
            print(output[i].shape)
            print(label[i].shape)
            example = serialize_example(output[i], label[i])
            writer.write(example)

            if step == 0:
                fig, axes = plt.subplots(1, 3, figsize=(12, 4))
                lr_show = np.transpose(lr[i], (1, 2, 0))
                axes[0].imshow(2*lr_show[:, :, [0,3,2]]*1+1)
                axes[0].set_title('Low-Resolution')
                axes[0].axis('off')

                output_show = np.transpose(output[i], (1, 2, 0))
                axes[1].imshow(2*(output_show[:, :, 3:0:-1]*0.0000275-0.2))
                axes[1].set_title('High-Resolution')
                axes[1].axis('off')

                axes[2].imshow(label[i])
                axes[2].set_title('Output')
                axes[2].axis('off')

                plt.show()

    # Close writer
    writer.close()
    

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1065
num_training = 852
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_River.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/BiDiff_x16_M2L8_River_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_River_BiDiff_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 762
num_training = 610
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CDL.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/BiDiff_x16_M2L8_CDL_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CDL_BiDiff_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1687
num_training = 1350
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_Urban.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/BiDiff_x16_M2L8_Urban_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_Urban_BiDiff_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)


In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 755
num_training = 604
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_GPP.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/BiDiff_x16_M2L8_GPP_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_GPP_BiDiff_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256  # 4x sr image size
label_size = 1000 #Vermontlc
class_num = 1
num_sample = 1408
num_training = 1126
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CHM.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/BiDiff_x16_M2L8_CHM_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CHM_BiDiff_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)